# Experiment 5: Subword-Level Tokenization, POS Tagging and Word2Vec

**Subject:** Computational Linguistics and Natural Language Processing

## Aim
To implement BPE and SentencePiece tokenization, POS tagging using spaCy and NLTK, and Word2Vec CBOW/Skip-Gram models with cosine similarity.

This notebook is designed for **Google Colab**. Run the cells from top to bottom. The notebook includes the provided text corpus, so you do not need to upload separate files.

> Note: Pretrained tokenizers and language models may need an internet connection on their first download. Actual token IDs, segmentation, vectors, and similarity scores are generated when the code runs.


## 1. Install and import libraries

In [ ]:
!pip -q install transformers tokenizers sentencepiece spacy nltk gensim
!python -m spacy download en_core_web_sm -q


In [ ]:
import os
import re
import nltk
import spacy
import sentencepiece as spm

from collections import Counter
from transformers import AutoTokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from gensim.models import Word2Vec

print("Libraries imported successfully.")


## 2. Prepare the input text

The following text is the supplied `input_sub_word_data.txt` corpus. We write it to a file so the tokenizer-training examples can use it directly.

In [ ]:
input_text = """Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is simple and
easy to understand, but it may have difficulty handling rare and unknown
words.

Subword-level tokenization provides an effective solution to the problem
of rare words. Instead of representing every complete word as a separate
token, a subword tokenizer represents a word using smaller meaningful units.
For example, the word playing can be represented as play and ing. Similarly,
the word unhappy can be represented as un and happy.

Byte Pair Encoding is a popular subword tokenization algorithm. BPE starts
with a vocabulary containing individual characters and repeatedly merges
the most frequent pair of adjacent symbols. The merging process continues
until the desired vocabulary size is obtained. Frequent character sequences
gradually become larger subword units.

BPE is useful because it provides a balance between word-level and
character-level tokenization. Common words can be represented as complete
tokens, while rare words can be divided into smaller subword units. This
allows the tokenizer to represent words that were not frequently observed
during training.

Machine learning algorithms require numerical representations of text.
After tokenization, each token can be assigned a unique numerical token ID.
These token IDs can then be converted into vectors and supplied to machine
learning or deep learning models.

Deep learning has significantly improved natural language processing.
Neural networks can learn complex patterns from large collections of text.
Recurrent neural networks, long short-term memory networks, convolutional
neural networks, and transformer models have all been used for language
processing tasks.

Transformers use attention mechanisms to identify relationships between
different tokens in a sequence. The attention mechanism allows a model to
consider important words even when they occur far apart in a sentence.
Transformer-based models have become widely used for text classification,
machine translation, question answering, summarization, and text generation.

Language models learn statistical relationships between tokens. A language
model can predict the next token based on the tokens that appear before it.
Modern language models are trained on very large text corpora containing
books, articles, websites, documents, and other forms of written language.

Consider the word internationalization. A word-level tokenizer may treat
internationalization as one token. A subword tokenizer can divide it into
smaller units such as international, ization, or other learned subword
segments. The exact segmentation depends on the vocabulary and the BPE
merge rules learned from the training corpus.

Similarly, words such as computational, communication, representation,
classification, information, educational, environmental, technological,
development, understanding, processing, and programming contain recurring
subword patterns. BPE can learn these recurring patterns from the corpus.

For example, the suffixes ing, ed, tion, ment, ness, and able may occur
frequently in a corpus. BPE can learn frequent sequences containing these
characters and use them as subword tokens. Prefixes such as un, re, pre,
dis, and con may also appear frequently and can become useful subword units.

A larger corpus generally produces more useful BPE vocabulary because the
algorithm has more examples from which to learn frequent character pairs.
A small corpus may produce vocabulary that is specific to the sentences
used for training. Therefore, a representative corpus is important when
building a practical tokenizer.

In an NLP experiment, the corpus can be converted to lowercase and divided
into words before learning BPE merges. Each word can initially be represented
as a sequence of characters followed by an end-of-word marker. The algorithm
then counts adjacent symbol pairs and identifies the most frequent pair.

The most frequent pair is merged into a new symbol. After every merge, the
pair frequencies are recalculated. This process is repeated several times.
The resulting collection of symbols forms the learned subword vocabulary.

Suppose a corpus contains many occurrences of the sequence t and h.
The BPE algorithm may first learn the pair th. If the sequence th occurs
frequently with e, the algorithm may subsequently learn the sequence the.
With additional training data, longer frequently occurring sequences can
be learned.

BPE is therefore a data-driven tokenization technique. It does not require
a manually constructed list of prefixes, suffixes, or linguistic rules.
Instead, the vocabulary is learned automatically from the statistical
properties of the training corpus.

Subword tokenization is particularly useful for multilingual applications.
Different languages may have different word formation patterns, and a
word-level vocabulary can become extremely large. Subword units can reduce
the vocabulary size while still allowing the tokenizer to represent many
different words.

Text preprocessing can include converting characters to lowercase,
removing unnecessary spaces, normalizing punctuation, and separating
sentences. However, preprocessing decisions should be selected according
to the objective of the NLP application.

A BPE tokenizer can also be evaluated by examining the number of unique
tokens, vocabulary size, number of merges, average tokens per word, and
frequency of individual subword units. These measurements help students
understand how the tokenizer behaves as the number of BPE merges changes.

In this laboratory experiment, the above corpus can be used as training
data for implementing BPE without using a pretrained tokenizer. Students
can experiment with different numbers of merges and compare the resulting
subword vocabularies.

The tokenizer can then be tested on new words such as naturalization,
internationalization, computationally, unbelievable, preprocessing,
classification, communication, renewable, hydrokinetic, multilingual,
understanding, and modernization.

The purpose of this experiment is to understand how BPE learns reusable
subword units and how they can be converted into token IDs."""

with open("input_sub_word_data.txt", "w", encoding="utf-8") as f:
    f.write(input_text)

print("Input corpus saved.")
print("Number of characters:", len(input_text))


# 5.1 BPE-based subword tokenization

BPE repeatedly merges frequent adjacent character/symbol pairs. A pretrained tokenizer already has a learned vocabulary; a tokenizer trained from scratch learns its vocabulary from our text.

### 5.1(a) Using a pretrained BPE tokenizer

In [ ]:
# GPT-2 uses byte-level BPE
bpe_pretrained = AutoTokenizer.from_pretrained("gpt2")

sample_text = "Natural language processing is interesting"

tokens = bpe_pretrained.tokenize(sample_text)
token_ids = bpe_pretrained.encode(sample_text, add_special_tokens=False)

print("Original text:", sample_text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("\nToken and ID:")
for token, token_id in zip(tokens, token_ids):
    print(f"{token:20} : {token_id}")


### 5.1(b) Without a pretrained tokenizer

In [ ]:
# Build and train a new BPE tokenizer from the supplied file
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()

bpe_trainer = BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["[UNK]"]
)

bpe_tokenizer.train(["input_sub_word_data.txt"], bpe_trainer)

sample_text = "Natural language processing is interesting"
encoded = bpe_tokenizer.encode(sample_text)

print("Original text:", sample_text)
print("Tokens:", encoded.tokens)
print("Token IDs:", encoded.ids)
print("\nToken and ID:")
for token, token_id in zip(encoded.tokens, encoded.ids):
    print(f"{token:20} : {token_id}")


# 5.2 SentencePiece tokenization

SentencePiece learns subword units directly from raw text. It commonly marks word boundaries using `▁`. Here, BPE is selected as the SentencePiece model type.

### 5.2(a) Using a pretrained SentencePiece tokenizer

In [ ]:
# T5 uses a pretrained SentencePiece tokenizer
sp_pretrained = AutoTokenizer.from_pretrained("t5-small")

sample_text = "Natural language processing is interesting"
tokens = sp_pretrained.tokenize(sample_text)
token_ids = sp_pretrained.encode(sample_text, add_special_tokens=False)

print("Original text:", sample_text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("\nToken and ID:")
for token, token_id in zip(tokens, token_ids):
    print(f"{token:20} : {token_id}")


### 5.2(b) Without a pretrained model

In [ ]:
# SentencePiece vocab size must be suitable for this corpus.
# A small value helps avoid errors with a small training dataset.
spm.SentencePieceTrainer.train(
    input="input_sub_word_data.txt",
    model_prefix="my_sentencepiece",
    vocab_size=300,
    model_type="bpe",
    character_coverage=1.0,
    hard_vocab_limit=False
)

sp_processor = spm.SentencePieceProcessor(
    model_file="my_sentencepiece.model"
)

sample_text = "Natural language processing is interesting"
tokens = sp_processor.encode(sample_text, out_type=str)
token_ids = sp_processor.encode(sample_text, out_type=int)

print("Original text:", sample_text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("\nToken and ID:")
for token, token_id in zip(tokens, token_ids):
    print(f"{token:20} : {token_id}")


# 5.3 POS tagging using spaCy and NLTK

POS tagging assigns grammatical categories such as noun, verb, adjective, and adverb to tokens.

### 5.3(a) Using spaCy

In [ ]:
# Load spaCy's English model
nlp = spacy.load("en_core_web_sm")

sentence = "The young student is reading an interesting book in the library."
doc = nlp(sentence)

print(f"{'Token':15} {'POS':10} Description")
seen = set()

for token in doc:
    key = token.text.lower()
    if token.is_alpha and key not in seen:
        seen.add(key)
        print(f"{token.text:15} {token.pos_:10} {spacy.explain(token.pos_)}")


### 5.3(b) Using NLTK

In [ ]:
# Download NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("universal_tagset")

sentence = "The young student is reading an interesting book in the library."
words = nltk.word_tokenize(sentence)
tagged_words = nltk.pos_tag(words, tagset="universal")

descriptions = {
    "NOUN": "Noun",
    "VERB": "Verb",
    "ADJ": "Adjective",
    "ADV": "Adverb",
    "PRON": "Pronoun",
    "DET": "Determiner",
    "ADP": "Preposition",
    "CONJ": "Conjunction",
    "PRT": "Particle",
    "NUM": "Numeral",
    "X": "Other",
    ".": "Punctuation"
}

print(f"{'Token':15} {'POS':10} Description")
seen = set()

for word, tag in tagged_words:
    key = word.lower()
    if word.isalpha() and key not in seen:
        seen.add(key)
        print(f"{word:15} {tag:10} {descriptions.get(tag, tag)}")


# 5.4 POS tagging with frequency using spaCy

This section processes the supplied text file, prints each unique lowercase word, its POS tag and description, and how often it occurs. The tag shown is from the word's first occurrence in the corpus.

In [ ]:
with open("input_sub_word_data.txt", "r", encoding="utf-8") as file:
    corpus_text = file.read()

doc = nlp(corpus_text)
frequency = Counter()
pos_info = {}

for token in doc:
    if token.is_alpha:
        word = token.text.lower()
        frequency[word] += 1
        if word not in pos_info:
            pos_info[word] = (token.pos_, spacy.explain(token.pos_))

print(f"{'Token':22} {'POS':10} {'Description':22} Frequency")
for word, count in frequency.items():
    pos, description = pos_info[word]
    print(f"{word:22} {pos:10} {description:22} {count}")


# 5.5 Word2Vec CBOW

CBOW (Continuous Bag of Words) predicts a target word from its surrounding context words. In Gensim, `sg=0` selects CBOW.

### Prepare the corpus

In [ ]:
corpus = [
    "Natural language processing is interesting",
    "Natural language processing is useful",
    "I love machine learning",
    "Machine learning is useful",
    "Deep learning is a part of machine learning",
    "Artificial intelligence is changing the world",
    "Natural language processing uses machine learning",
    "Deep learning and artificial intelligence are related",
    "I study natural language processing",
    "Machine learning helps artificial intelligence"
]

with open("corpus.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(corpus))

sentences = [line.lower().split() for line in corpus]
print("Corpus prepared. Number of sentences:", len(sentences))


### Train CBOW and inspect a word

In [ ]:
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    workers=1,
    seed=42,
    epochs=100
)

word = "language"
print("Vector representation of", word)
print(cbow_model.wv[word])

print("\nThree most similar words:")
for similar_word, score in cbow_model.wv.most_similar(word, topn=3):
    print(similar_word, ":", round(score, 4))


# 5.6 Word2Vec Skip-Gram

Skip-Gram predicts surrounding context words from a target word. In Gensim, `sg=1` selects Skip-Gram.

In [ ]:
skipgram_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    seed=42,
    epochs=100
)

word = "language"
print("Vector representation of", word)
print(skipgram_model.wv[word])

print("\nFive most similar words:")
for similar_word, score in skipgram_model.wv.most_similar(word, topn=5):
    print(similar_word, ":", round(score, 4))


# 5.7 Cosine similarity using Word2Vec

Cosine similarity compares the directions of two vectors. Gensim's `similarity()` method calculates cosine similarity between the selected word embeddings.

In [ ]:
similarity_pairs = [
    ("machine", "learning"),
    ("natural", "language"),
    ("deep", "learning")
]

for word1, word2 in similarity_pairs:
    score = cbow_model.wv.similarity(word1, word2)
    print(f"Similarity between {word1} and {word2}: {score:.4f}")


# Result

The notebook implements:
- BPE tokenization with a pretrained tokenizer and a tokenizer trained from scratch.
- SentencePiece tokenization with a pretrained tokenizer and a newly trained model.
- POS tagging using spaCy and NLTK.
- POS tagging with token frequencies using spaCy.
- Word2Vec CBOW and Skip-Gram training.
- Cosine similarity for the specified word pairs.

# Conclusion

Subword tokenization divides text into smaller units and maps them to token IDs. POS tagging identifies the grammatical role of words. Word2Vec learns vector representations from word context, and cosine similarity compares the resulting vectors.

**Note:** The supplied corpus is small. Word2Vec similarity results are therefore only demonstrations of the method and should not be treated as reliable general-purpose semantic measurements.